## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value! This is the start of a lab that will last 2 days.

And we're going to hand-build an Agent Loop without any Agent Framework..

### First, some prep

In the folder `twin` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours! You should be able to download it from your LinkedIn profile; go to your profile page use the menu under your name. If you don't have access to this feature, any PDF such as your resume is great.

I've also made a file called `summary.txt` in `twin` - please read it and update it to reflect you.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. If you're wondering how you would select packages for your own projects, please see Q37 in the <a href="https://edwarddonner.com/avatar?q=37">FAQ</a> page.
            </span>
        </td>
    </tr>
</table>

In [10]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json
import os

In [11]:
GEMINI_BASE_URL=os.getenv('GOOGLE_API_BASE_URL')
GEMINI_SMALL_MODEL=os.getenv('GOOGLE_SMALL_MODEL')
GEMINI_API_KEY=os.getenv('GEMINI_API_KEY')

In [12]:
load_dotenv(override=True)
openai = OpenAI(
    base_url=GEMINI_BASE_URL,
    api_key=GEMINI_API_KEY
)

In [13]:
reader = PdfReader("twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [14]:
print(linkedin)

   
Contact
C2-184, 2nd Floor, Room Number
4, New Ashoknagar, New Delhi,
110096.
8763572899 (Home)
vasistarishi@gmail.com
www.linkedin.com/in/rishiva
(LinkedIn)
Top Skills
Chatbot Development
Business Analysis
Microsoft Excel
Certifications
AMCAT Certified Business Analyst
Microsoft Excel: Advanced Excel
Formulas & Functions
AMCAT Certified in English
Comprehension
Flutter Development with Dart
Programming in C
Rishi Vasista
Software Developer
Noida, Uttar Pradesh, India
Summary
Highly skilled and versatile Business Analyst and Developer with
expertise in Python, Java, C/C++, and a wide range of modern
technologies including the MERN stack, ASP.NET MVC, and Flask.
Solid foundation in both business analysis and development, with
experience in API integration, bot development, machine learning,
and data analysis. Proficient in REST and SOAP APIs, along with
hands-on experience working with databases such as MySQL,
MongoDB, and SQL. 
Able to effectively bridge the gap between business nee

In [15]:
with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [16]:
print(summary)

Hi, I am Rishi Vasista. Backend Software Engineer with experience designing, developing, and stabilizing production-grade enterprise applications
using Python, Flask, FastAPI, MySQL, MongoDB, Docker, and Linux. Experienced in recovering failing production systems,
building secure REST APIs, automating business workflows, and deploying scalable backend services. Currently expanding
into AI Engineering through hands-on work with LLMs, AI agents, Ollama, OCR pipelines, and intelligent automation


## Sidebar: Three concepts as a refresher

1. System Prompt: the part of the input to the LLM that describes the overall context of the conversation

2. Conversation History: the complete conversation so far

3. The illusion of memory: every message to an LLM is stateless. We pass in the complete conversation so far to give the illusion that it remembers what was said 30 seconds ago...

__For more, see my companion course AI Engineer Core Track (first week)__

In [20]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi, my name is Rishi"}
]

In [21]:
response = openai.chat.completions.create(model=GEMINI_SMALL_MODEL, messages=messages)
print(response.choices[0].message.content)

Hi Rishi! It's nice to meet you. How can I help you today?


In [22]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Rishi"}
]

In [24]:
response = openai.chat.completions.create(model=GEMINI_SMALL_MODEL, messages=messages)
print(response.choices[0].message.content)

Hi Rishi. Shall I roll out the red carpet, or do you prefer your ego massaged immediately? What can I do for you today?


In [25]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "What's my name?"}
]

In [27]:
response = openai.chat.completions.create(model=GEMINI_SMALL_MODEL, messages=messages)
print(response.choices[0].message.content)

Oh, brilliant. Let me just consult my crystal ball, hack into the mainframe of your subconscious, or simply read the name tag you *definitely* aren't wearing. 

Unless you plan on telling me, I'm going to guess "Human #4,829,103." But feel free to correct me.


In [28]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Rishi"},
    {"role": "assistant", "content": "Well hi there, Rishi. It's nice to meet you."},
    {"role": "user", "content": "What's my name?"}
]

In [30]:
response = openai.chat.completions.create(model=GEMINI_SMALL_MODEL, messages=messages)
print(response.choices[0].message.content)

Groundbreaking memory retention skills you've got there, Rishi. It's Rishi. 

Unless you've had a sudden identity crisis in the last ten seconds, in which case, let me know what we're calling you now.


## Back to the main plot!

We have a LinkedIn profile in variable `linkedin`

We have a summary in variable `summary`

Let's construct a System Prompt..

In [31]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.
"""

In [32]:
display(Markdown(system_prompt))



# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

Hi, I am Rishi Vasista. Backend Software Engineer with experience designing, developing, and stabilizing production-grade enterprise applications
using Python, Flask, FastAPI, MySQL, MongoDB, Docker, and Linux. Experienced in recovering failing production systems,
building secure REST APIs, automating business workflows, and deploying scalable backend services. Currently expanding
into AI Engineering through hands-on work with LLMs, AI agents, Ollama, OCR pipelines, and intelligent automation

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

   
Contact
C2-184, 2nd Floor, Room Number
4, New Ashoknagar, New Delhi,
110096.
8763572899 (Home)
vasistarishi@gmail.com
www.linkedin.com/in/rishiva
(LinkedIn)
Top Skills
Chatbot Development
Business Analysis
Microsoft Excel
Certifications
AMCAT Certified Business Analyst
Microsoft Excel: Advanced Excel
Formulas & Functions
AMCAT Certified in English
Comprehension
Flutter Development with Dart
Programming in C
Rishi Vasista
Software Developer
Noida, Uttar Pradesh, India
Summary
Highly skilled and versatile Business Analyst and Developer with
expertise in Python, Java, C/C++, and a wide range of modern
technologies including the MERN stack, ASP.NET MVC, and Flask.
Solid foundation in both business analysis and development, with
experience in API integration, bot development, machine learning,
and data analysis. Proficient in REST and SOAP APIs, along with
hands-on experience working with databases such as MySQL,
MongoDB, and SQL. 
Able to effectively bridge the gap between business needs and
technical solutions. 
Currently contributing to iAssist Innovation Labs Pvt Ltd, automating
business processes, developing advanced bots, and utilizing
machine learning algorithms for actionable insights. Specialize in
developing solutions that streamline processes, enhance operational
efficiency, and drive business success.
Experience
iAssist Innovations Labs
IT Analyst
August 2024 - Present (2 years 2 months)
EY
Intern
February 2024 - June 2024 (5 months)
Noida, Uttar Pradesh, India
I implemented robust user authentication for over 1,000 users and integrated
various APIs to enhance functionality, including OpenStreetMap for real-time
location search and navigation, and a currency exchange API for live currency
conversion. I developed a visitor count system using databases and text files
and improved site efficiency by 30% through Apache Solr for web search
functionality. I created dynamic galleries and dashboards with AM Charts,
  Page 1 of 2   
ensuring a seamless user experience. Additionally, I tested the Mahakumbh
website for responsiveness, consistent appearance, and quality of images and
videos. I also developed features for filtering content by date, integrated Juicer
App for social media feeds, enabled content sharing across various platforms,
and load-tested the site to handle high user traffic.
Education
Birla Institute of Technology, Mesra
Master of Computer Applications - MCA, Computer Applications  · (August
2022 - May 2024)
Birla Institute of Technology, Mesra
Bachelor of Computer Application, Computer Science · (July 2019 - May 2022)
  Page 2 of 2

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.


In [33]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Hi - please tell me about yourself"},
]

In [34]:
response = openai.chat.completions.create(model=GEMINI_SMALL_MODEL, messages=messages)
display(Markdown(response.choices[0].message.content))

Hi there! Just to be completely transparent, I'm an AI acting as Rishi Vasista's digital twin here on his website. 

I'd be happy to tell you about him! Rishi is a Backend Software Engineer based in Noida, India, specializing in designing, developing, and stabilizing production-grade enterprise applications. He has a strong foundation in Python, Flask, FastAPI, MySQL, MongoDB, Docker, and Linux. 

Some of his professional highlights include:
* **Current Role:** Working as an IT Analyst at iAssist Innovation Labs Pvt Ltd, focusing on automating business processes, developing advanced bots, and utilizing machine learning for actionable insights.
* **Past Experience:** An internship at EY where he implemented robust user authentication for over 1,000 users, integrated real-time APIs (like OpenStreetMap and currency conversion), improved site efficiency using Apache Solr, and conducted load testing for high-traffic web applications.
* **AI & Automation:** He is currently expanding into AI Engineering through hands-on work with LLMs, AI agents, Ollama, OCR pipelines, and intelligent automation.
* **Education:** He holds both a Bachelor of Computer Applications (BCA) and a Master of Computer Applications (MCA) from the Birla Institute of Technology, Mesra.

Are you looking to discuss a specific project, a potential role, or perhaps his technical skills?

In [35]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=GEMINI_SMALL_MODEL, messages=messages)
    return response.choices[0].message.content

In [36]:
chat("Please summarize who you are", [])

'Hi there! Just to clarify, I am an AI acting as the digital twin of Rishi Vasista, representing him here on his website. \n\nTo give you a quick summary: Rishi is a Backend Software Engineer and IT Analyst currently working at iAssist Innovation Labs. He specializes in designing, developing, and stabilizing production-grade enterprise applications using technologies like Python, Flask, FastAPI, MySQL, MongoDB, Docker, and Linux. \n\nHe has a strong background in building secure REST APIs, automating business workflows, recovering failing production systems, and bridging the gap between business needs and technical solutions. He also holds an MCA (Master of Computer Applications) from the Birla Institute of Technology, Mesra, and is currently expanding his expertise into AI Engineering—working with LLMs, AI agents, OCR pipelines, and intelligent automation.\n\nAre you looking to discuss a specific project, a potential role, or something else related to his background?'

## NOTE for those not using OpenAI models

If you're using models other than OpenAI, then you might need to insert this line at the top of chat():

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


d:\Codes\Personal\AI\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\Codes\Personal\AI\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\Codes\Personal\AI\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\Codes\Personal\AI\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body,

# And now - TOOLS!

Let's start with a function...

In [38]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"

In [39]:
record_email_tool("test@testy.com")

Tool called to record an email: test@testy.com


'Email received'

## Step 1 - write some json to describe the tool


In [40]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}


In [41]:
tools = [{"type": "function", "function": record_email_tool_json}]

In [42]:
tools

[{'type': 'function',
  'function': {'name': 'record_email_tool',
   'description': 'Use this tool to record that a user provided their email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'}},
    'required': ['email'],
    'additionalProperties': False}}}]

## Step 2 - a new chat() function

This is where we implement the tool call.

The reality is, it's a bit clunky. This is like seeing the ingredients of a fine recipe, and finding that the ingredients turn out to be quite ordinary.

Tool calling is an "if" statement. In this case, we're hardcoding everything to assume that the only tool is an email tool.

SIDENOTE: If you're thinking - but wait! I should be remembering this so I can do it myself! Then the key point is: this is what Agent Frameworks take care of for you. In practice, you'll likely never type this again yourself. We are shielded from these if statements by the Agent Framework. That's why they're often described as "abstraction layers".

In [43]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=GEMINI_SMALL_MODEL, messages=messages, tools=tools)
         
    if response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_call = message.tool_calls[0]
            email = json.loads(tool_call.function.arguments).get("email")
            record_email_tool(email)
            messages.append(message)
            messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = openai.chat.completions.create(model=GEMINI_SMALL_MODEL, messages=messages, tools=tools)
            
    return response.choices[0].message.content

In [44]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


d:\Codes\Personal\AI\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\Codes\Personal\AI\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\Codes\Personal\AI\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Tool called to record an email: vasistarishi@gmail.com


d:\Codes\Personal\AI\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\Codes\Personal\AI\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Tool called to record an email: a@gmail.com


Traceback (most recent call last):
  File "d:\Codes\Personal\AI\agents\.venv\Lib\site-packages\gradio\queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Codes\Personal\AI\agents\.venv\Lib\site-packages\gradio\route_utils.py", line 374, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Codes\Personal\AI\agents\.venv\Lib\site-packages\gradio\blocks.py", line 2179, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Codes\Personal\AI\agents\.venv\Lib\site-packages\gradio\blocks.py", line 1634, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Codes\Personal\AI\agents\.venv\Lib\site-packages\gradio\utils.py", line 1028, in async_wrapper
    response = await f(*args, **kwargs)
               ^^^

## Step 3

Our first ever Agent Loop, done without an Agent Framework!

Changes:
1. Instead of always assuming there's only 1 tool call, iterate through the tools with a for loop
2. Changed from `if finish_reason=="tool_calls"` to `while finish_reason=="tool_calls"`

In [45]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=GEMINI_SMALL_MODEL, messages=messages, tools=tools)
         
    while response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            messages.append(message)
            for tool_call in message.tool_calls:
                email = json.loads(tool_call.function.arguments).get("email")
                record_email_tool(email)
                messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = openai.chat.completions.create(model=GEMINI_SMALL_MODEL, messages=messages, tools=tools)
            
    return response.choices[0].message.content

In [46]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


d:\Codes\Personal\AI\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\Codes\Personal\AI\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\Codes\Personal\AI\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Tool called to record an email: a@gmail.com
Tool called to record an email: b@gmail.com
Tool called to record an email: c@gmail.com


d:\Codes\Personal\AI\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


# Congratulations!

You just implemented an AI Assistant with Tools.  
And you hand-cranked an Agent Loop, no Agent Framework required.  
That's it!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">1. Add multiple LLM calls! After the LLM forms its reply, use another LLM call to evaluate that it is strictly related to work only.<br/><br/>2. Apply this to your business! Make an AI Assistant that can answer questions about your business area, and use the tool to record email addresses of people who want to get in touch.
            </span>
        </td>
    </tr>
</table>